In [3]:
!pip install firebase-admin


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [10]:
import json
import re
import firebase_admin
from firebase_admin import credentials, firestore

# 1. Inicialização
SERVICE_ACCOUNT = "/Users/leosaracino/Documents/Pessoal/Faculdade/TCC/motiva-8b82f-firebase-adminsdk-fbsvc-14d8d2b5e8.json"
try:
    firebase_admin.get_app()
except ValueError:
    cred = credentials.Certificate(SERVICE_ACCOUNT)
    firebase_admin.initialize_app(cred)

db = firestore.client()

# 2. Carrega e normaliza o JSON
with open('exercises.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    exercises = data
elif isinstance(data, dict) and 'Exercicios' in data and isinstance(data['Exercicios'], dict):
    exercises = list(data['Exercicios'].values())
elif isinstance(data, dict):
    exercises = list(data.values())
else:
    raise ValueError("Formato JSON não suportado. Deve ser array de objetos ou dict de dicts.")

# 3. Gerador de IDs (snake_case) caso falte
def slugify(name: str) -> str:
    s = name.strip().lower()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    return s.strip('_')

# 4. Importação com verificação de existência
for ex in exercises:
    # garante displayName
    disp = ex.get('displayName')
    if not disp:
        print("⚠️  Exercício sem displayName, pulando:", ex)
        continue

    # ID do doc
    doc_id = ex.get('name') or slugify(disp)
    ex['name'] = doc_id

    doc_ref = db.collection('Exercicios').document(doc_id)
    if doc_ref.get().exists:
        print(f"⏭️  Já existe, pulando: {doc_id}")
        continue

    # grava
    try:
        doc_ref.set(ex)
        print(f"✔️  Importado: {doc_id}")
    except Exception as e:
        print(f"❌ Falha ao importar {doc_id}: {e}")

print("✨ Importação concluída!")


✔️  Importado: air_squat
✔️  Importado: front_squat
✔️  Importado: overhead_squat
✔️  Importado: shoulder_press
✔️  Importado: push_press
✔️  Importado: push_jerk
✔️  Importado: deadlift
✔️  Importado: sumo_deadlift_high_pull
✔️  Importado: snatch
✔️  Importado: clean_and_jerk
✔️  Importado: thruster
✔️  Importado: kettlebell_swing
✔️  Importado: wall_ball_shot
✔️  Importado: pull_up
✔️  Importado: chest_to_bar_pull_up
✔️  Importado: kipping_pull_up
✔️  Importado: butterfly_pull_up
✔️  Importado: muscle_up
✔️  Importado: ring_row
✔️  Importado: handstand_push_up
✔️  Importado: toes_to_bar
✔️  Importado: ghd_sit_up
✔️  Importado: box_jump
✔️  Importado: double_unders
✔️  Importado: burpee
✔️  Importado: lunges
✔️  Importado: rowing
✔️  Importado: assault_bike
✔️  Importado: farmers_carry
✔️  Importado: turkish_get_up
✔️  Importado: pistols
✨ Importação concluída!
